# Multi-Class Prediction of Cirrhosis Outcomes - Karaciğer Sirozu Sonuçlarının Çok Sınıflı Tahmini

<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcS0krTPDAE1yQOtXil4EViQcrCBw-549P42PgMrpBAD&s">

Bu projede, siroz hastalarının klinik ve laboratuvar verileri kullanılarak hastaların sağlık durumlarının (C, CL veya D) tahmin edilmesi amaçlanmıştır. Veri seti üzerinde gerekli ön işleme adımları uygulanmış, kategorik değişkenler sayısallaştırılmış ve makine öğrenmesi algoritmalarının kullanımı için veri hazır hale getirilmiştir. Ardından farklı sınıflandırma modelleri eğitilerek performansları Accuracy, Precision, Recall, F1 Score ve Log Loss metrikleri ile karşılaştırılmış, veri seti için en başarılı model belirlenmiştir.

### Sütun Açıklamaları

**id**: Her satır için benzersiz kimlik numarasıdır.

**N_Days**: Hastanın takip edildiği gün sayısını gösterir.

**Drug**: Hastaya verilen tedavi türünü gösterir. Örneğin D-penicillamine veya Placebo.

**Age**: Hastanın yaşını gün cinsinden gösterir.

**Sex**: Hastanın cinsiyetini gösterir. M erkek, F kadın anlamındadır.

**Ascites**: Hastada karında sıvı birikimi olup olmadığını gösterir. Y var, N yok.

**Hepatomegaly**: Hastada karaciğer büyümesi olup olmadığını gösterir. Y var, N yok.

**Spiders**: Hastada örümcek damar görünümü olup olmadığını gösterir. Y var, N yok.

**Edema**: Hastada ödem durumunu gösterir. N yok, S tedaviyle düzelen ödem, Y tedaviye rağmen devam eden ödem.

**Bilirubin**: Kandaki bilirubin seviyesini gösterir.

**Cholesterol**: Kandaki kolesterol seviyesini gösterir.

**Albumin**: Kandaki albümin seviyesini gösterir.

**Copper**: İdrardaki bakır seviyesini gösterir.

**Alk_Phos**: Alkalen fosfataz enzim seviyesini gösterir.

**SGOT**: Karaciğer fonksiyonlarıyla ilişkili SGOT/AST enzim seviyesini gösterir.

**Tryglicerides**: Kandaki trigliserid seviyesini gösterir.

**Platelets**: Kandaki trombosit sayısını gösterir.

**Prothrombin**: Protrombin süresini gösterir.

**Stage**: Hastalığın evresini gösterir.

**Status**: Hedef değişkendir. Hastanın son durumunu gösterir. C takipte olay yok, CL karaciğer nakli nedeniyle takip sonlandı, D ölüm.

### Veri seti linki

https://www.kaggle.com/competitions/playground-series-s3e26/data

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s3e26/sample_submission.csv
/kaggle/input/competitions/playground-series-s3e26/train.csv
/kaggle/input/competitions/playground-series-s3e26/test.csv


In [2]:
# ==========================================
# Veri Analizi
# ==========================================
import pandas as pd
import numpy as np

# ==========================================
# Görselleştirme
# ==========================================
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# Uyarıları Kapatma
# ==========================================
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# Ön İşleme
# ==========================================
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# ==========================================
# Modeller
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

# ==========================================
# Değerlendirme Metrikleri
# ==========================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# ==========================================
# Çapraz Doğrulama (İsteğe Bağlı)
# ==========================================
from sklearn.model_selection import cross_val_score

In [3]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)
    for filename in filenames:
        print("   ", filename)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/playground-series-s3e26
    sample_submission.csv
    train.csv
    test.csv


In [4]:
# Veri setlerini oku
train = pd.read_csv("/kaggle/input/competitions/playground-series-s3e26/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s3e26/test.csv")

# İlk 5 satırı görüntüle
train.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,999,D-penicillamine,21532,M,N,N,N,N,2.3,316.0,3.35,172.0,1601.0,179.80,63.0,394.0,9.7,3.0,D
1,1,2574,Placebo,19237,F,N,N,N,N,0.9,364.0,3.54,63.0,1440.0,134.85,88.0,361.0,11.0,3.0,C
2,2,3428,Placebo,13727,F,N,Y,Y,Y,3.3,299.0,3.55,131.0,1029.0,119.35,50.0,199.0,11.7,4.0,D
3,3,2576,Placebo,18460,F,N,N,N,N,0.6,256.0,3.50,58.0,1653.0,71.30,96.0,269.0,10.7,3.0,C
4,4,788,Placebo,16658,F,N,Y,N,N,1.1,346.0,3.65,63.0,1181.0,125.55,96.0,298.0,10.6,4.0,C


In [5]:
train.shape

(7905, 20)

In [6]:
train.columns

Index(['id', 'N_Days', 'Drug', 'Age', 'Sex', 'Ascites', 'Hepatomegaly',
       'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper',
       'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin',
       'Stage', 'Status'],
      dtype='object')

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7905 entries, 0 to 7904
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             7905 non-null   int64  
 1   N_Days         7905 non-null   int64  
 2   Drug           7905 non-null   object 
 3   Age            7905 non-null   int64  
 4   Sex            7905 non-null   object 
 5   Ascites        7905 non-null   object 
 6   Hepatomegaly   7905 non-null   object 
 7   Spiders        7905 non-null   object 
 8   Edema          7905 non-null   object 
 9   Bilirubin      7905 non-null   float64
 10  Cholesterol    7905 non-null   float64
 11  Albumin        7905 non-null   float64
 12  Copper         7905 non-null   float64
 13  Alk_Phos       7905 non-null   float64
 14  SGOT           7905 non-null   float64
 15  Tryglicerides  7905 non-null   float64
 16  Platelets      7905 non-null   float64
 17  Prothrombin    7905 non-null   float64
 18  Stage   

In [9]:
train.isnull().sum()

id               0
N_Days           0
Drug             0
Age              0
Sex              0
Ascites          0
Hepatomegaly     0
Spiders          0
Edema            0
Bilirubin        0
Cholesterol      0
Albumin          0
Copper           0
Alk_Phos         0
SGOT             0
Tryglicerides    0
Platelets        0
Prothrombin      0
Stage            0
Status           0
dtype: int64

In [10]:
train.describe()

,id,N_Days,Age,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
count,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000,7905.000000
mean,3952.000000,2030.173308,18373.146490,2.594485,350.561923,3.548323,83.902846,1816.745250,114.604602,115.340164,265.228969,10.629462,3.032511
std,2282.121272,1094.233744,3679.958739,3.812960,195.379344,0.346171,75.899266,1903.750657,48.790945,52.530402,87.465579,0.781735,0.866511
min,0.000000,41.000000,9598.000000,0.300000,120.000000,1.960000,4.000000,289.000000,26.350000,33.000000,62.000000,9.000000,1.000000
25%,1976.000000,1230.000000,15574.000000,0.700000,248.000000,3.350000,39.000000,834.000000,75.950000,84.000000,211.000000,10.000000,2.000000
50%,3952.000000,1831.000000,18713.000000,1.100000,298.000000,3.580000,63.000000,1181.000000,108.500000,104.000000,265.000000,10.600000,3.000000
75%,5928.000000,2689.000000,20684.000000,3.000000,390.000000,3.770000,102.000000,1857.000000,137.950000,139.000000,316.000000,11.000000,4.000000
max,7904.000000,4795.000000,28650.000000,28.000000,1775.000000,4.640000,588.000000,13862.400000,457.250000,598.000000,563.000000,18.000000,4.000000


In [11]:
print(train["Drug"].unique())

['D-penicillamine' 'Placebo']


In [12]:
drug_mapping = {
    "Placebo": 0,
    "D-penicillamine": 1
}

train["Drug"] = train["Drug"].map(drug_mapping)
test["Drug"] = test["Drug"].map(drug_mapping)

train["Drug"].head()

0    1
1    0
2    0
3    0
4    0
Name: Drug, dtype: int64

In [14]:
sex_mapping = {
    "F": 0,
    "M": 1
}

train["Sex"] = train["Sex"].map(sex_mapping)
test["Sex"] = test["Sex"].map(sex_mapping)

train["Sex"].head()

0    1
1    0
2    0
3    0
4    0
Name: Sex, dtype: int64

In [16]:
ascites_mapping = {
    "N": 0,
    "Y": 1
}

train["Ascites"] = train["Ascites"].map(ascites_mapping)
test["Ascites"] = test["Ascites"].map(ascites_mapping)

train["Ascites"].head()

0    0
1    0
2    0
3    0
4    0
Name: Ascites, dtype: int64

In [17]:
hepatomegaly_mapping = {
    "N": 0,
    "Y": 1
}

train["Hepatomegaly"] = train["Hepatomegaly"].map(hepatomegaly_mapping)
test["Hepatomegaly"] = test["Hepatomegaly"].map(hepatomegaly_mapping)

train["Hepatomegaly"].head()

0    0
1    0
2    1
3    0
4    1
Name: Hepatomegaly, dtype: int64

In [18]:
spiders_mapping = {
    "N": 0,
    "Y": 1
}

train["Spiders"] = train["Spiders"].map(spiders_mapping)
test["Spiders"] = test["Spiders"].map(spiders_mapping)

train["Spiders"].head()

0    0
1    0
2    1
3    0
4    0
Name: Spiders, dtype: int64

In [19]:
print(train["Edema"].unique())

['N' 'Y' 'S']


In [20]:
edema_mapping = {
    "N": 0,
    "S": 1,
    "Y": 2
}

train["Edema"] = train["Edema"].map(edema_mapping)
test["Edema"] = test["Edema"].map(edema_mapping)

train["Edema"].head()

0    0
1    0
2    2
3    0
4    0
Name: Edema, dtype: int64

In [21]:
print(train["Status"].unique())

['D' 'C' 'CL']


In [22]:
status_mapping = {
    "C": 0,
    "CL": 1,
    "D": 2
}

train["Status"] = train["Status"].map(status_mapping)

train["Status"].head()

0    2
1    0
2    2
3    0
4    0
Name: Status, dtype: int64

In [23]:
train.head()

,id,N_Days,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Status
0,0,999,1,21532,1,0,0,0,0,2.3,316.0,3.35,172.0,1601.0,179.80,63.0,394.0,9.7,3.0,2
1,1,2574,0,19237,0,0,0,0,0,0.9,364.0,3.54,63.0,1440.0,134.85,88.0,361.0,11.0,3.0,0
2,2,3428,0,13727,0,0,1,1,2,3.3,299.0,3.55,131.0,1029.0,119.35,50.0,199.0,11.7,4.0,2
3,3,2576,0,18460,0,0,0,0,0,0.6,256.0,3.50,58.0,1653.0,71.30,96.0,269.0,10.7,3.0,0
4,4,788,0,16658,0,0,1,0,0,1.1,346.0,3.65,63.0,1181.0,125.55,96.0,298.0,10.6,4.0,0


In [24]:
train = train.drop("id", axis=1)
test = test.drop("id", axis=1)

In [26]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7905 entries, 0 to 7904
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   N_Days         7905 non-null   int64  
 1   Drug           7905 non-null   int64  
 2   Age            7905 non-null   int64  
 3   Sex            7905 non-null   int64  
 4   Ascites        7905 non-null   int64  
 5   Hepatomegaly   7905 non-null   int64  
 6   Spiders        7905 non-null   int64  
 7   Edema          7905 non-null   int64  
 8   Bilirubin      7905 non-null   float64
 9   Cholesterol    7905 non-null   float64
 10  Albumin        7905 non-null   float64
 11  Copper         7905 non-null   float64
 12  Alk_Phos       7905 non-null   float64
 13  SGOT           7905 non-null   float64
 14  Tryglicerides  7905 non-null   float64
 15  Platelets      7905 non-null   float64
 16  Prothrombin    7905 non-null   float64
 17  Stage          7905 non-null   float64
 18  Status  

In [27]:
# Bağımsız değişkenler (Features)
x = train.drop("Status", axis=1)

# Bağımlı değişken (Target)
y = train["Status"]

In [30]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

In [32]:
# Random Forest

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(x_train, y_train)

y_pred_rf = rf_model.predict(x_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Accuracy: 0.8317520556609741


In [33]:
# Decision Tree

dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(x_train, y_train)

y_pred_dt = dt_model.predict(x_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

Decision Tree Accuracy: 0.7394054395951929


In [34]:
# Logistic Regression

lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(x_train, y_train)

y_pred_lr = lr_model.predict(x_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.767235926628716


In [35]:
# KNN

knn_model = KNeighborsClassifier()

knn_model.fit(x_train, y_train)

y_pred_knn = knn_model.predict(x_test)

print("KNN Accuracy:", accuracy_score(y_test, y_pred_knn))

KNN Accuracy: 0.7609108159392789


In [36]:
# Naive Bayes

nb_model = GaussianNB()

nb_model.fit(x_train, y_train)

y_pred_nb = nb_model.predict(x_test)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.7526881720430108


In [37]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss
)

import pandas as pd

def algo_test(x, y):

    LR = LogisticRegression(max_iter=1000)
    DT = DecisionTreeClassifier(random_state=42)
    RF = RandomForestClassifier(random_state=42)
    ET = ExtraTreesClassifier(random_state=42)
    GB = GradientBoostingClassifier(random_state=42)
    NB = GaussianNB()

    algos = [LR, DT, RF, ET, GB, NB]

    algo_names = [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "Extra Trees",
        "Gradient Boosting",
        "Naive Bayes"
    ]

    x_train, x_test, y_train, y_test = train_test_split(
        x,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    result = pd.DataFrame(
        columns=[
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
            "Log Loss"
        ],
        index=algo_names
    )

    accuracy = []
    precision = []
    recall = []
    f1 = []
    logloss = []

    for algo in algos:

        algo.fit(x_train, y_train)

        p = algo.predict(x_test)
        proba = algo.predict_proba(x_test)

        accuracy.append(accuracy_score(y_test, p))

        precision.append(
            precision_score(y_test, p, average="weighted")
        )

        recall.append(
            recall_score(y_test, p, average="weighted")
        )

        f1.append(
            f1_score(y_test, p, average="weighted")
        )

        logloss.append(
            log_loss(y_test, proba)
        )

    result["Accuracy"] = accuracy
    result["Precision"] = precision
    result["Recall"] = recall
    result["F1 Score"] = f1
    result["Log Loss"] = logloss

    return result.sort_values(
        "Log Loss",
        ascending=True
    )

In [38]:
algo_test(x, y)

,Accuracy,Precision,Recall,F1 Score,Log Loss
Gradient Boosting,0.824794,0.816775,0.824794,0.816483,0.459644
Random Forest,0.831752,0.829277,0.831752,0.818991,0.568201
Extra Trees,0.829222,0.817877,0.829222,0.817115,0.581176
Logistic Regression,0.767236,0.736139,0.767236,0.748348,0.601184
Naive Bayes,0.752688,0.747790,0.752688,0.742585,1.595163
Decision Tree,0.739405,0.737095,0.739405,0.738225,9.392780


## Sonuç

Farklı makine öğrenmesi algoritmalarının performansları karşılaştırıldığında, ensemble tabanlı modellerin diğer yöntemlere göre daha başarılı sonuçlar verdiği görülmüştür. Özellikle Gradient Boosting, en düşük Log Loss değeri ile en iyi performansı sergilemiştir. Random Forest ve Extra Trees modelleri de başarılı sonuçlar elde ederken, Decision Tree modeli en düşük performansı göstermiştir. Genel olarak, bu veri seti için en uygun modelin Gradient Boosting olduğu değerlendirilmiştir.